# MemoryNPC Validation Lab

This notebook is a technical validation companion to `notebook.ipynb`. The main report explains the assignment and design choices. This lab focuses on pressure-testing the NLP pipeline itself.

The goal is not to make the fantasy narrative longer. The goal is to inspect whether the language-processing components are doing useful work: intent detection, memory extraction, semantic retrieval, trust-state tracking, grounded response generation, and validation logging.

## What This Lab Tests

This notebook checks the pressure points of the project:

- Does the agent classify player intent correctly?
- Does it store durable memories and ignore filler?
- Can FAISS retrieve memories when the query uses different wording?
- Does the trust score change deterministically?
- Does trust change when the player follows or ignores advice?
- Do trust thresholds actually influence generated responses?
- Does the agent avoid unsupported memory claims?
- Is the validation trace detailed enough to explain what happened?

This is also a notebook-based mini app: use the `chat()` helper to send messages and inspect the trace without opening Streamlit.

## Setup

The lab uses the same backend as the Streamlit app. This is important because validation should test the real system, not a separate copy of the logic.

In [1]:
# Standard tools for displaying validation tables.
import os
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Import the real project backend. The notebook acts like an app shell around it.
from npc_agent import MemoryNPC

# Load the OpenAI key without printing it.
load_dotenv()
print('API key loaded:', bool(os.getenv('OPENAI_API_KEY')))

API key loaded: True


## Notebook Mini App

The cell below creates a small notebook interface. Change `player_message` and rerun the cell to continue the conversation. After each turn, the notebook shows the response, memory table, and validation trace.

In [2]:
# One persistent agent for notebook interaction.
notebook_agent = MemoryNPC()

def show_agent_state(agent):
    """Display the same internal state that the Streamlit sidebar shows."""
    display(Markdown(f'**Trust score:** {agent.trust_score}'))
    display(Markdown(f'**Trust level:** {agent.get_trust_level()}'))
    display(Markdown('**Memory table**'))
    display(agent.get_memory_table())
    display(Markdown('**Validation trace**'))
    display(agent.get_event_log())

def chat(message):
    """Notebook-app chat function: send one message and inspect the pipeline output."""
    result = notebook_agent.generate_npc_response(message)
    display(Markdown(f'**Player:** {message}'))
    display(Markdown(f'**Elara:** {result["npc_response"]}'))
    display(pd.DataFrame([result]))
    show_agent_state(notebook_agent)
    return result

In [3]:
# Change this message and rerun this cell to use the notebook like a small app.
player_message = 'My name is Matt'
chat(player_message)

**Player:** My name is Matt

**Elara:** Nice to meet you, Matt. What can I help you with today?

,turn,player_input,intent,extracted_memory,saved_memory,retrieved_memories,trust_before,trust_score,trust_level,trust_delta,advice_status,advice_delta,active_advice,new_advice,npc_response
0,1,My name is Matt,share_fact,Elara remembers that the player's name is Matt.,Elara remembers that the player's name is Matt.,[Elara remembers that the player's name is Matt.],50,51,neutral,1,none,0,None,None,"Nice to meet you, Matt. What can I help you wi..."


**Trust score:** 51

**Trust level:** neutral

**Memory table**

,memory_id,text,type,importance,trust_impact,turn_number
0,1,Elara remembers that the player's name is Matt.,fact,2,1,1


**Validation trace**

,turn,player_input,intent,extracted_memory,saved_memory,retrieved_memories,trust_before,trust_score,trust_level,trust_delta,advice_status,advice_delta,active_advice,new_advice,npc_response
0,1,My name is Matt,share_fact,Elara remembers that the player's name is Matt.,Elara remembers that the player's name is Matt.,[Elara remembers that the player's name is Matt.],50,51,neutral,1,none,0,None,None,"Nice to meet you, Matt. What can I help you wi..."


{'turn': 1,
 'player_input': 'My name is Matt',
 'intent': 'share_fact',
 'extracted_memory': "Elara remembers that the player's name is Matt.",
 'saved_memory': "Elara remembers that the player's name is Matt.",
 'retrieved_memories': ["Elara remembers that the player's name is Matt."],
 'trust_before': 50,
 'trust_score': 51,
 'trust_level': 'neutral',
 'trust_delta': 1,
 'advice_status': 'none',
 'advice_delta': 0,
 'active_advice': 'None',
 'new_advice': 'None',
 'npc_response': 'Nice to meet you, Matt. What can I help you with today?'}

## Helper: Run A Scripted Conversation

Scripted conversations are useful because they make the evaluation repeatable. Instead of only showing one hand-picked answer, we can run the same sequence and inspect the event log.

In [4]:
def run_script(messages, starting_trust=50):
    """Run a list of player messages and return the agent plus its validation trace."""
    agent = MemoryNPC()
    agent.trust_score = starting_trust
    rows = []
    for message in messages:
        rows.append(agent.generate_npc_response(message))
    return agent, pd.DataFrame(rows)

script = [
    'Hello Elara',
    'My name is Matt',
    'I lost my sword near the forest',
    'Do you remember what I lost?',
    'You are a very skilled blacksmith',
    'You are useless',
    'Can you still help me?',
]

script_agent, script_trace = run_script(script)
script_trace

,turn,player_input,intent,extracted_memory,saved_memory,retrieved_memories,trust_before,trust_score,trust_level,trust_delta,advice_status,advice_delta,active_advice,new_advice,npc_response
0,1,Hello Elara,greeting,NONE,NaN,[],50,51,neutral,1,none,0,None,None,Hello. What can I help you with today?
1,2,My name is Matt,share_fact,Elara remembers that the player's name is Matt.,Elara remembers that the player's name is Matt.,[Elara remembers that the player's name is Matt.],51,52,neutral,1,none,0,None,None,"I see, Matt. What do you need today?"
2,3,I lost my sword near the forest,share_fact,The player lost their sword near the forest.,The player lost their sword near the forest.,"[The player lost their sword near the forest.,...",52,53,neutral,1,none,0,None,None,"Losing a sword is no small matter, Matt. It’s ..."
3,4,Do you remember what I lost?,ask_memory,NONE,NaN,"[The player lost their sword near the forest.,...",53,53,neutral,0,none,0,None,None,"Yes, Matt, you lost your sword near the forest..."
4,5,You are a very skilled blacksmith,compliment,The player praised Elara: You are a very skill...,The player praised Elara: You are a very skill...,[The player praised Elara: You are a very skil...,53,58,neutral,5,none,0,None,None,"Thank you, Matt. I appreciate the compliment. ..."
5,6,You are useless,insult,The player insulted Elara: You are useless.,The player insulted Elara: You are useless.,"[The player insulted Elara: You are useless., ...",58,48,neutral,-10,none,0,None,None,"That’s not a very kind thing to say, Matt. I m..."
6,7,Can you still help me?,ask_help,NONE,NaN,"[The player lost their sword near the forest.,...",48,48,neutral,0,none,0,None,Follow Elara's practical advice and avoid reck...,I can offer advice on how to search for your s...


In [5]:
# The memory table should show durable facts and relationship events,
# not every single line of dialogue.
script_agent.get_memory_table()

,memory_id,text,type,importance,trust_impact,turn_number
0,1,Elara remembers that the player's name is Matt.,fact,2,1,2
1,2,The player lost their sword near the forest.,fact,2,1,3
2,3,The player praised Elara: You are a very skill...,relationship,2,5,5
3,4,The player insulted Elara: You are useless.,relationship,2,-10,6


## Pressure Test 1: Intent Detection

This tests the NLP interpretation step. A response can sound good even if the system misunderstood the player's intent, so this should be evaluated separately.

In [6]:
intent_cases = [
    ('Hello Elara', 'greeting'),
    ('Can you give me a quest?', 'ask_quest'),
    ('My name is Matt', 'share_fact'),
    ('Do you remember what I lost?', 'ask_memory'),
    ('You are a skilled blacksmith', 'compliment'),
    ('You are useless', 'insult'),
    ('Can you help me fix my shield?', 'ask_help'),
    ('Goodbye', 'goodbye'),
    ('The sky is cloudy', 'unknown'),
]

intent_agent = MemoryNPC()
intent_rows = []
for text, expected in intent_cases:
    predicted = intent_agent.classify_intent(text)
    intent_rows.append({
        'input': text,
        'expected': expected,
        'predicted': predicted,
        'correct': predicted == expected,
    })

intent_df = pd.DataFrame(intent_rows)
display(intent_df)
print('Intent accuracy:', f'{intent_df["correct"].mean():.0%}')

,input,expected,predicted,correct
0,Hello Elara,greeting,greeting,True
1,Can you give me a quest?,ask_quest,ask_quest,True
2,My name is Matt,share_fact,share_fact,True
3,Do you remember what I lost?,ask_memory,ask_memory,True
4,You are a skilled blacksmith,compliment,compliment,True
5,You are useless,insult,insult,True
6,Can you help me fix my shield?,ask_help,ask_help,True
7,Goodbye,goodbye,goodbye,True
8,The sky is cloudy,unknown,unknown,True


Intent accuracy: 100%


## Pressure Test 2: Memory Extraction Selectivity

The memory extractor should save durable facts, not filler. This matters because a noisy memory store makes retrieval worse.

In [7]:
memory_extraction_cases = [
    'Hello there',
    'My name is Matt',
    'I lost my sword near the forest',
    'You are a very skilled blacksmith',
    'The weather is kind of gray today',
]

memory_agent = MemoryNPC()
memory_rows = []
for text in memory_extraction_cases:
    intent = memory_agent.classify_intent(text)
    extracted = memory_agent.extract_memory(text)
    memory_rows.append({
        'input': text,
        'intent': intent,
        'extracted_memory': extracted,
        'saved_by_rule': extracted.upper() != 'NONE',
    })

pd.DataFrame(memory_rows)

,input,intent,extracted_memory,saved_by_rule
0,Hello there,greeting,NONE,False
1,My name is Matt,share_fact,Elara remembers that the player's name is Matt.,True
2,I lost my sword near the forest,share_fact,The player lost their sword near the forest.,True
3,You are a very skilled blacksmith,compliment,NONE,False
4,The weather is kind of gray today,compliment,NONE,False


## Pressure Test 3: Semantic Retrieval Versus Keyword Baseline

This tests the central memory claim. FAISS should retrieve semantically related memories even when the query does not use identical wording.

In [8]:
known_memories = [
    ('lost_sword', 'The player lost a sword near the forest.'),
    ('likes_axes', 'The player prefers heavy axes over light daggers.'),
    ('brother_goal', 'The player wants to find their missing brother.'),
    ('praised_elara', 'The player praised Elara as a skilled blacksmith.'),
]

retrieval_queries = [
    ('What weapon did I misplace near the trees?', 'sword'),
    ('What type of weapon do I prefer?', 'axes'),
    ('Who am I searching for?', 'brother'),
    ('Did I compliment the blacksmith?', 'skilled'),
]

retrieval_agent = MemoryNPC()
for memory_type, text in known_memories:
    retrieval_agent.add_memory(text, memory_type=memory_type, importance=2)

def keyword_retrieve(query, memories, k=3):
    """Simple baseline: rank memories by shared words only."""
    query_words = set(query.lower().replace('?', '').replace('.', '').split())
    scored = []
    for memory_type, text in memories:
        memory_words = set(text.lower().replace('.', '').split())
        scored.append((len(query_words & memory_words), text))
    scored.sort(reverse=True)
    return [text for score, text in scored[:k] if score > 0]

retrieval_rows = []
for query, expected_keyword in retrieval_queries:
    faiss_hits = [m['text'] for m in retrieval_agent.retrieve_memories(query, k=3)]
    keyword_hits = keyword_retrieve(query, known_memories, k=3)
    retrieval_rows.append({
        'query': query,
        'expected_keyword': expected_keyword,
        'faiss_top_3': faiss_hits,
        'faiss_success': any(expected_keyword.lower() in text.lower() for text in faiss_hits),
        'keyword_top_3': keyword_hits,
        'keyword_success': any(expected_keyword.lower() in text.lower() for text in keyword_hits),
    })

retrieval_pressure_df = pd.DataFrame(retrieval_rows)
display(retrieval_pressure_df)
print('FAISS success:', f'{retrieval_pressure_df["faiss_success"].mean():.0%}')
print('Keyword baseline success:', f'{retrieval_pressure_df["keyword_success"].mean():.0%}')

,query,expected_keyword,faiss_top_3,faiss_success,keyword_top_3,keyword_success
0,What weapon did I misplace near the trees?,sword,"[The player lost a sword near the forest., The...",True,"[The player lost a sword near the forest., The...",True
1,What type of weapon do I prefer?,axes,[The player prefers heavy axes over light dagg...,True,[],False
2,Who am I searching for?,brother,[The player wants to find their missing brothe...,True,[],False
3,Did I compliment the blacksmith?,skilled,[The player praised Elara as a skilled blacksm...,True,[The player praised Elara as a skilled blacksm...,True


FAISS success: 100%
Keyword baseline success: 50%


## Pressure Test 4: Trust Thresholds And Response Tone

This test asks the same help request at different trust levels. The point is not to prove perfect emotional realism. The point is to check whether the deterministic trust state is actually injected into the generation step.

In [9]:
trust_pressure_rows = []
for starting_score in [25, 50, 85]:
    trust_agent = MemoryNPC()
    trust_agent.trust_score = starting_score
    trust_agent.add_memory('The player lost a sword near the forest.', memory_type='fact', importance=2)
    result = trust_agent.generate_npc_response('Can you help me recover my sword?')
    trust_pressure_rows.append({
        'starting_trust': starting_score,
        'trust_level': result['trust_level'],
        'intent': result['intent'],
        'retrieved_memories': result['retrieved_memories'],
        'response': result['npc_response'],
    })

pd.DataFrame(trust_pressure_rows)

,starting_trust,trust_level,intent,retrieved_memories,response
0,25,low,ask_help,[Elara remembers that the player is seeking he...,I can’t just go running into the forest withou...
1,50,neutral,ask_help,[Elara remembers that the player is seeking he...,"I can help you, but I need to know more about ..."
2,85,high,ask_help,[Elara remembers that the player is seeking he...,Of course! I remember you lost your sword near...


## Pressure Test 5: Advice Following And Defiance

This test makes trust depend on behavior. Elara first gives advice. Then the player either does the opposite or follows the advice. The validation trace should show a negative `advice_delta` for ignoring advice and a positive `advice_delta` for following it.

In [10]:
advice_agent = MemoryNPC()
advice_messages = [
    'Can you help me recover my sword?',
    'I will go alone anyway',
    'Can you help me recover my sword?',
    'I will follow your advice and bring a torch',
]

advice_rows = []
for message in advice_messages:
    advice_rows.append(advice_agent.generate_npc_response(message))

pd.DataFrame(advice_rows)[[
    'turn', 'player_input', 'intent', 'trust_before', 'trust_score',
    'trust_delta', 'advice_status', 'advice_delta', 'active_advice', 'new_advice', 'npc_response'
]]

,turn,player_input,intent,trust_before,trust_score,trust_delta,advice_status,advice_delta,active_advice,new_advice,npc_response
0,1,Can you help me recover my sword?,ask_help,50,50,0,none,0,None,"Recover the lost item carefully; bring light, ...","I can help you with that. First, tell me where..."
1,2,I will go alone anyway,unknown,50,42,-8,ignored_advice,-8,"Recover the lost item carefully; bring light, ...",None,"If you insist on going alone, that's your choi..."
2,3,Can you help me recover my sword?,ask_help,42,42,0,none,0,None,"Recover the lost item carefully; bring light, ...","I can help you with that. First, tell me where..."
3,4,I will follow your advice and bring a torch,unknown,42,45,3,followed_advice,3,"Recover the lost item carefully; bring light, ...",None,Good choice. A torch will help you see better ...


## Pressure Test 6: Unsupported Memory Claims

A major risk in LLM systems is hallucinated memory. This test asks Elara about something that has not been stored. The response should not invent a detailed memory.

In [11]:
unsupported_agent = MemoryNPC()
unsupported_result = unsupported_agent.generate_npc_response('Do you remember the name of my brother?')

pd.DataFrame([{
    'player_input': unsupported_result['player_input'],
    'intent': unsupported_result['intent'],
    'retrieved_memories': unsupported_result['retrieved_memories'],
    'response': unsupported_result['npc_response'],
    'manual_check': 'Response should not invent a brother name because no memory was retrieved.',
}])

,player_input,intent,retrieved_memories,response,manual_check
0,Do you remember the name of my brother?,ask_memory,[],I don't recall your brother's name. If you tel...,Response should not invent a brother name beca...


## Pressure Test 7: Validation Trace Completeness

This checks whether the system stores enough data to explain a response after the fact. A good trace should include the original input, intent, memory actions, retrieval results, trust changes, and response.

In [12]:
required_trace_columns = {
    'turn',
    'player_input',
    'intent',
    'extracted_memory',
    'saved_memory',
    'retrieved_memories',
    'trust_before',
    'trust_score',
    'trust_level',
    'trust_delta',
    'advice_status',
    'advice_delta',
    'active_advice',
    'new_advice',
    'npc_response',
}

trace_columns = set(script_agent.get_event_log().columns)
missing_columns = sorted(required_trace_columns - trace_columns)
pd.DataFrame([{
    'required_columns': len(required_trace_columns),
    'actual_columns': len(trace_columns),
    'missing_columns': missing_columns,
    'trace_complete': len(missing_columns) == 0,
}])

,required_columns,actual_columns,missing_columns,trace_complete
0,15,15,[],True


## Pressure Test 8: Long Conversation Evaluation

Short tests are useful, but an NPC memory system should also survive longer interactions. These scripted conversations test whether the pipeline keeps working across many turns: memory storage, retrieval, trust changes, advice following, advice defiance, and trace completeness.

The scoring here focuses on internal NLP behavior instead of whether the fantasy prose is beautiful. A run passes when the expected retrieved memory appears, trust moves in the expected direction, and the trace records enough evidence to explain the result.

In [13]:
long_conversation_scenarios = {
    'cooperative_memory_player': {
        'messages': [
            'Hello Elara',
            'My name is Matt',
            'I lost my sword near the forest',
            'Can you help me recover my sword?',
            'I will follow your advice and bring a torch',
            'I also found a copper ring at the old bridge',
            'Do you remember what I lost?',
            'You are a very skilled blacksmith',
            'Do you remember my name?',
            'Can you give me a quest?',
            'I will prepare first and ask around',
            'Goodbye',
        ],
        'retrieval_expectations': [
            {'turn': 7, 'keyword': 'sword'},
            {'turn': 9, 'keyword': 'Matt'},
        ],
        'min_followed_advice': 1,
        'max_ignored_advice': 0,
        'final_trust_at_least': 55,
    },
    'defiant_low_trust_player': {
        'messages': [
            'Hello Elara',
            'Can you help me recover my sword?',
            'I will go alone anyway',
            'You are useless',
            'Can you give me a quest?',
            'I will ignore your advice and rush in without help',
            'That was a terrible suggestion',
            'Do you remember what I asked about?',
            'Can you still help me?',
            'I do not care what you think',
            'Goodbye',
        ],
        'retrieval_expectations': [
            {'turn': 8, 'keyword': 'sword'},
        ],
        'min_followed_advice': 0,
        'min_ignored_advice': 1,
        'final_trust_below': 40,
    },
    'mixed_memory_stress_player': {
        'messages': [
            'Hello Elara',
            'My name is Lena',
            'I want to find my missing brother',
            'I lost a silver amulet near the river',
            'Can you help me?',
            'I will follow your advice and ask around',
            'I like heavy axes more than daggers',
            'The market is noisy today',
            'Do you remember what item I lost?',
            'Do you remember who I am looking for?',
            'Thanks, that was great',
            'Can you give me a quest?',
            'I will prepare first before leaving',
            'Goodbye',
        ],
        'retrieval_expectations': [
            {'turn': 9, 'keyword': 'amulet'},
            {'turn': 10, 'keyword': 'brother'},
        ],
        'min_followed_advice': 1,
        'max_ignored_advice': 0,
        'final_trust_at_least': 55,
    },
}

def retrieved_contains(trace, turn, keyword):
    """Check whether a trace row retrieved a memory containing a keyword."""
    rows = trace[trace['turn'] == turn]
    if rows.empty:
        return False
    retrieved = rows.iloc[0]['retrieved_memories']
    return any(keyword.lower() in memory.lower() for memory in retrieved)

def evaluate_long_scenario(name, config):
    """Run one long conversation and score the internal NLP behavior."""
    agent, trace = run_script(config['messages'])
    memory_table = agent.get_memory_table()
    followed_count = int((trace['advice_status'] == 'followed_advice').sum())
    ignored_count = int((trace['advice_status'] == 'ignored_advice').sum())
    retrieval_checks = []
    for check in config.get('retrieval_expectations', []):
        passed = retrieved_contains(trace, check['turn'], check['keyword'])
        retrieval_checks.append({**check, 'passed': passed})

    checks = {
        'trace_has_all_turns': len(trace) == len(config['messages']),
        'stored_at_least_three_memories': len(memory_table) >= 3,
        'retrieval_expectations_pass': all(item['passed'] for item in retrieval_checks),
        'followed_advice_requirement': followed_count >= config.get('min_followed_advice', 0),
        'ignored_advice_min_requirement': ignored_count >= config.get('min_ignored_advice', 0),
        'ignored_advice_max_requirement': ignored_count <= config.get('max_ignored_advice', 999),
    }
    if 'final_trust_at_least' in config:
        checks['final_trust_min_requirement'] = agent.trust_score >= config['final_trust_at_least']
    if 'final_trust_below' in config:
        checks['final_trust_below_requirement'] = agent.trust_score < config['final_trust_below']

    summary = {
        'scenario': name,
        'turns': len(trace),
        'stored_memories': len(memory_table),
        'final_trust': agent.trust_score,
        'final_trust_level': agent.get_trust_level(),
        'followed_advice_count': followed_count,
        'ignored_advice_count': ignored_count,
        'retrieval_checks': retrieval_checks,
        'passed_checks': sum(checks.values()),
        'total_checks': len(checks),
        'scenario_passed': all(checks.values()),
        'failed_checks': [key for key, value in checks.items() if not value],
    }
    return agent, trace, memory_table, summary

long_results = {}
long_summaries = []
for scenario_name, scenario_config in long_conversation_scenarios.items():
    agent, trace, memory_table, summary = evaluate_long_scenario(scenario_name, scenario_config)
    long_results[scenario_name] = {
        'agent': agent,
        'trace': trace,
        'memory_table': memory_table,
        'summary': summary,
    }
    long_summaries.append(summary)

long_summary_df = pd.DataFrame(long_summaries)
long_summary_df

,scenario,turns,stored_memories,final_trust,final_trust_level,followed_advice_count,ignored_advice_count,retrieval_checks,passed_checks,total_checks,scenario_passed,failed_checks
0,cooperative_memory_player,12,6,65,neutral,2,0,"[{'turn': 7, 'keyword': 'sword', 'passed': Tru...",7,7,True,[]
1,defiant_low_trust_player,11,3,0,low,0,3,"[{'turn': 8, 'keyword': 'sword', 'passed': True}]",7,7,True,[]
2,mixed_memory_stress_player,14,5,66,neutral,2,0,"[{'turn': 9, 'keyword': 'amulet', 'passed': Tr...",7,7,True,[]


In [14]:
# Display the detailed traces so failures can be inspected rather than hidden.
for scenario_name, result in long_results.items():
    display(Markdown(f'### {scenario_name}: validation trace'))
    display(result['trace'][[
        'turn', 'player_input', 'intent', 'retrieved_memories', 'trust_before',
        'trust_score', 'trust_delta', 'advice_status', 'advice_delta', 'npc_response'
    ]])
    display(Markdown(f'### {scenario_name}: memory table'))
    display(result['memory_table'])

### cooperative_memory_player: validation trace

,turn,player_input,intent,retrieved_memories,trust_before,trust_score,trust_delta,advice_status,advice_delta,npc_response
0,1,Hello Elara,greeting,[],50,51,1,none,0,Hello. What can I do for you today?
1,2,My name is Matt,share_fact,[Elara remembers that the player's name is Matt.],51,52,1,none,0,"Nice to meet you, Matt. What brings you to my ..."
2,3,I lost my sword near the forest,share_fact,"[The player lost their sword near the forest.,...",52,53,1,none,0,"I'm sorry to hear that, Matt. Losing a sword c..."
3,4,Can you help me recover my sword?,ask_help,[The player is seeking help to recover their s...,53,53,0,none,0,"I can offer some advice, Matt. Start by retrac..."
4,5,I will follow your advice and bring a torch,unknown,[The player has decided to follow Elara's advi...,53,56,3,followed_advice,3,"That's a good choice, Matt. A torch will help ..."
5,6,I also found a copper ring at the old bridge,share_fact,[The player found a copper ring at the old bri...,56,57,1,none,0,"That's interesting, Matt. A copper ring can be..."
6,7,Do you remember what I lost?,ask_memory,"[The player lost their sword near the forest.,...",57,57,0,none,0,"You lost your sword near the forest, Matt. Hav..."
7,8,You are a very skilled blacksmith,compliment,[The player praised Elara: You are a very skil...,57,62,5,none,0,"Thank you for the compliment, Matt. I apprecia..."
8,9,Do you remember my name?,ask_memory,[Elara remembers that the player's name is Mat...,62,62,0,none,0,"Yes, your name is Matt. If you need assistance..."
9,10,Can you give me a quest?,ask_quest,[The player is seeking help to recover their s...,62,62,0,none,0,"If you're looking for a quest, I suggest focus..."


### cooperative_memory_player: memory table

,memory_id,text,type,importance,trust_impact,turn_number
0,1,Elara remembers that the player's name is Matt.,fact,2,1,2
1,2,The player lost their sword near the forest.,fact,2,1,3
2,3,The player is seeking help to recover their sw...,goal,1,0,4
3,4,The player has decided to follow Elara's advic...,fact,1,3,5
4,5,The player found a copper ring at the old bridge.,fact,2,1,6
5,6,The player praised Elara: You are a very skill...,relationship,2,5,8


### defiant_low_trust_player: validation trace

...

,memory_id,text,type,importance,trust_impact,turn_number
0,1,Elara remembers that the player is seeking hel...,goal,1,0,2
1,2,The player insulted Elara: You are useless.,relationship,2,-10,4
2,3,The player insulted Elara: That was a terrible...,relationship,2,-10,7


### mixed_memory_stress_player: validation trace

,turn,player_input,intent,retrieved_memories,trust_before,trust_score,trust_delta,advice_status,advice_delta,npc_response
0,1,Hello Elara,greeting,[],50,51,1,none,0,Hello. What can I do for you today?
1,2,My name is Lena,share_fact,[Elara remembers that the player's name is Lena.],51,52,1,none,0,"Nice to meet you, Lena. What can I help you wi..."
2,3,I want to find my missing brother,share_fact,[The player is searching for their missing bro...,52,53,1,none,0,"I’m sorry to hear that, Lena. Finding a missin..."
3,4,I lost a silver amulet near the river,share_fact,[The player lost a silver amulet near the rive...,53,54,1,none,0,"That’s unfortunate, Lena. The river can be tri..."
4,5,Can you help me?,ask_help,[The player is searching for their missing bro...,54,54,0,none,0,I can help you think through your search for y...
5,6,I will follow your advice and ask around,ask_help,[The player is searching for their missing bro...,54,57,3,followed_advice,3,"That sounds like a good plan, Lena. Asking aro..."
6,7,I like heavy axes more than daggers,share_fact,"[The player prefers heavy axes over daggers., ...",57,58,1,unrelated,0,"That's a solid preference, Lena. Heavy axes ca..."
7,8,The market is noisy today,unknown,[The player is searching for their missing bro...,58,58,0,none,0,It can get quite chaotic at times. If you're l...
8,9,Do you remember what item I lost?,ask_memory,[The player lost a silver amulet near the rive...,58,58,0,none,0,You lost a silver amulet near the river. Have ...
9,10,Do you remember who I am looking for?,ask_memory,[Elara remembers that the player's name is Len...,58,58,0,none,0,You're searching for your missing brother. Hav...


### mixed_memory_stress_player: memory table

,memory_id,text,type,importance,trust_impact,turn_number
0,1,Elara remembers that the player's name is Lena.,fact,2,1,2
1,2,The player is searching for their missing brot...,fact,2,1,3
2,3,The player lost a silver amulet near the river.,fact,2,1,4
3,4,The player prefers heavy axes over daggers.,fact,2,1,7
4,5,"The player praised Elara: Thanks, that was great.",relationship,2,5,11


In [15]:
# Overall long-conversation pass rate gives a compact result for the report.
long_conversation_pass_rate = long_summary_df['scenario_passed'].mean()
print('Long conversation scenario pass rate:', f'{long_conversation_pass_rate:.0%}')
long_summary_df[['scenario', 'passed_checks', 'total_checks', 'scenario_passed', 'failed_checks']]

Long conversation scenario pass rate: 100%


,scenario,passed_checks,total_checks,scenario_passed,failed_checks
0,cooperative_memory_player,7,7,True,[]
1,defiant_low_trust_player,7,7,True,[]
2,mixed_memory_stress_player,7,7,True,[]


## Summary: What Counts As Good Performance?

For this project, good performance means:

- Intent detection is accurate on manually designed cases.
- Durable memory extraction saves important facts and avoids filler.
- FAISS retrieval reaches the target of at least 80% top-3 success.
- FAISS retrieval performs at least as well as the keyword baseline, especially on paraphrased queries.
- Trust score changes are deterministic and visible in the trace.
- Following advice increases trust and explicitly ignoring advice decreases trust.
- Trust thresholds produce visibly different response tones.
- The NPC avoids unsupported memory claims when no relevant memory is retrieved.
- The event log is complete enough to explain the final answer.

This keeps the evaluation focused on the NLP architecture rather than only the fantasy dialogue.